
# Smart Irrigation — Data Analysis & Moisture Prediction Lab

**D-CoE Industry 4.0 Technologies Programme · AI/ML & Analytics module**

In the IoT lab you built a closed-loop irrigation bench and logged every reading the
ESP32 sent — moisture, pump state, mode, signal strength. This is the same telemetry,
in bulk, from a full cohort of benches across several workshop days.

By the end of this notebook you will have:

1. Cleaned real (simulated-but-realistic) sensor telemetry — glitches, gaps and all.
2. Built physical intuition for *why* moisture behaves the way it does, with a live
   playground you control with sliders.
3. Trained two models: one that learns a sensor's calibration, and one that
   **forecasts moisture 5 minutes into the future** — the same problem your bench's
   `dryThreshold`/`wetThreshold` logic solves with a fixed rule.
4. Checked whether the forecasting model still works on a bench it has never seen.
5. Played with a live "what-if" predictor to see exactly which inputs move the output,
   and by how much.

> **Files this notebook expects in the same folder:**
> `telemetry-training.csv` (benches 01–05, 3 sessions each) and
> `telemetry-holdout.csv` (bench-06 — held back on purpose, see Part 4).



## Data dictionary

This is the same schema your dashboard exports (`GET /api/data/export.csv`), with one
addition: a `sessionId` column, since this bundle covers many workshop sessions across
many benches and we need a way to group them. Your own single-session exports won't
have it.

| Column | Meaning |
|---|---|
| `timestamp` | UTC time of the reading |
| `deviceId` | which bench sent it (`bench-01` … `bench-06`) |
| `moistureRaw` | raw ADC value from the YL-69, 0–4095 |
| `moisturePercent` | the firmware's converted moisture reading, 0–100 |
| `pumpOn` | 1 if the relay was energised at that instant |
| `mode` | `auto` (threshold logic) or `manual` (dashboard override) |
| `rssi` | Wi-Fi signal strength, dBm |
| `uptimeMs` | milliseconds since the ESP32 last rebooted |
| `dryThreshold` / `wetThreshold` | the auto-mode thresholds configured for that bench |
| `sessionId` | *(added for this bundle)* which logged session the row belongs to |


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import cm
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# D-CoE palette, so every plot in this notebook reads as one family
FOREST  = "#0A1F14"
ACCENT  = "#1EB553"
SAGE    = "#CBE9D4"
AMBER   = "#C97B0A"
TEAL    = "#0E7490"
NAVY    = "#0D2B55"
MUTED   = "#5B6B62"

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"] = 11

pd.set_option("display.max_columns", 20)
np.random.seed(7)
print("ready")



---
## Part 1 — The moisture playground

Before we open a single CSV: soil moisture in this system obeys one simple rule —

> **moisture falls on its own (evaporation), and rises only when the pump runs —
> and the pump is switched by two thresholds, not a thermostat dial.**

Move the sliders below and watch how each knob changes the shape of the curve. This
*is* the system you wired on the bench — just sped up and made controllable, so the
patterns you're about to look for in real data are ones you've already seen with your
own hands on the sliders.


In [ ]:

def simulate_playground(evap_rate=0.0008, pump_rate=0.07, dry_th=35, wet_th=65,
                         sensor_noise=0.4, hours=2.0):
    '''A stripped-down version of the exact physics used to generate the real
    dataset below -- same equations, fewer moving parts, so you can isolate one
    knob at a time.'''
    dt = 10  # seconds
    n = int(hours * 3600 / dt)
    moisture = 50.0
    pump_on = 0
    M_MIN, M_MAX = 8.0, 92.0
    t, m, p = [], [], []
    for i in range(n):
        if moisture < dry_th:
            pump_on = 1
        elif moisture > wet_th:
            pump_on = 0
        d = -evap_rate * (moisture - M_MIN) * dt
        if pump_on:
            d += pump_rate * dt
        d += np.random.normal(0, 0.05)
        moisture = float(np.clip(moisture + d, M_MIN, M_MAX))
        reading = float(np.clip(moisture + np.random.normal(0, sensor_noise), 0, 100))
        t.append(i * dt / 60.0)   # minutes
        m.append(reading)
        p.append(pump_on)
    return np.array(t), np.array(m), np.array(p)


def plot_playground(evap_rate, pump_rate, dry_th, wet_th, sensor_noise):
    t, m, p = simulate_playground(evap_rate, pump_rate, dry_th, wet_th, sensor_noise)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.fill_between(t, 0, 100, where=p.astype(bool), color=ACCENT, alpha=0.12,
                     step="post", label="pump ON")
    ax.plot(t, m, color=FOREST, lw=1.6, label="moisture %")
    ax.axhline(dry_th, color="#C0392B", ls="--", lw=1, label="dryThreshold")
    ax.axhline(wet_th, color="#2471A3", ls="--", lw=1, label="wetThreshold")
    ax.set_ylim(0, 100)
    ax.set_xlabel("minutes")
    ax.set_ylabel("moisture %")
    ax.set_title("Turn a knob, watch the cycle change shape")
    ax.legend(loc="upper right", fontsize=9, framealpha=0.9)
    plt.show()
    duty = p.mean() * 100
    print(f"pump was ON {duty:.0f}% of the session  |  {int(p.sum()*10/60)} min of pumping")


interact(
    plot_playground,
    evap_rate=widgets.FloatSlider(value=0.0008, min=0.0003, max=0.0018, step=0.0001,
                                   readout_format=".4f", description="evap rate"),
    pump_rate=widgets.FloatSlider(value=0.07, min=0.02, max=0.15, step=0.005,
                                   description="pump rate"),
    dry_th=widgets.IntSlider(value=35, min=10, max=50, description="dryThreshold"),
    wet_th=widgets.IntSlider(value=65, min=50, max=90, description="wetThreshold"),
    sensor_noise=widgets.FloatSlider(value=0.4, min=0.0, max=3.0, step=0.1,
                                      description="sensor noise"),
);



**Try this before moving on:**
- Push `pump rate` well above `evap rate` — the ON phase gets short and sharp. Push it
  the other way and the pump barely keeps up.
- Widen the gap between `dryThreshold` and `wetThreshold` — the pump cycles less often
  but runs longer each time. This is exactly the trade-off you're making when you set
  thresholds on the real dashboard: cycle frequency vs. pump wear vs. how tightly
  moisture is held near a target.
- Crank `sensor noise` up — this is what a loose probe or a bad ground connection
  looks like in the data, and it's the first thing we'll go looking for below.



---
## Part 2 — Load the real telemetry

"Real" here means: generated by the same physical rules you just played with, from six
independently-wired benches, across several separate workshop sessions — with the
sensor noise, glitches and dropped packets you'd actually get off an ESP32 on a shared
Wi-Fi network. Nothing below has been cleaned yet.


In [ ]:

train_raw = pd.read_csv("telemetry-training.csv", parse_dates=["timestamp"])
holdout_raw = pd.read_csv("telemetry-holdout.csv", parse_dates=["timestamp"])

print("training set:", train_raw.shape)
print("holdout set: ", holdout_raw.shape)
print("\ntraining devices:", sorted(train_raw.deviceId.unique()))
print("holdout devices: ", sorted(holdout_raw.deviceId.unique()), " <- kept separate on purpose")
train_raw.head()


In [ ]:

train_raw.describe()



Look at the `moisturePercent` row in that table. The min/max should already look
wrong to you — moisture is a percentage. That's not a bug in this notebook, it's the
first thing real sensor data will do to you.



---
## Part 3 — Trust nothing until you've checked it

Two failure modes to hunt for, both realistic for a battery of ESP32s on a shared
classroom Wi-Fi:

1. **Sensor glitches** — a loose jumper or a brief disconnect pins the ADC at a rail
   (0 or 4095), and the firmware dutifully converts that garbage into an out-of-range
   `moisturePercent`.
2. **Dropped packets** — a reading never reaches the server, leaving a gap in the
   timestamp sequence instead of a bad row.


In [ ]:

glitch_mask = (train_raw.moisturePercent < 0) | (train_raw.moisturePercent > 100)
print(f"glitch rows: {glitch_mask.sum()}  ({glitch_mask.mean()*100:.2f}% of the data)")
train_raw.loc[glitch_mask, ["deviceId", "timestamp", "moistureRaw", "moisturePercent"]].head(10)


In [ ]:

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.scatter(train_raw.moistureRaw, train_raw.moisturePercent, s=4, alpha=0.15, color=FOREST)
ax.scatter(train_raw.loc[glitch_mask, "moistureRaw"], train_raw.loc[glitch_mask, "moisturePercent"],
           s=18, color="#C0392B", label="glitch (raw pinned at 0 / 4095)")
ax.set_xlabel("moistureRaw"); ax.set_ylabel("moisturePercent")
ax.set_title("Every glitch traces back to a pinned raw value")
ax.legend()
plt.show()


In [ ]:

# gaps: how many timestamps are missing per device+session, relative to the expected 10s cadence
def expected_vs_actual(g):
    span = (g.timestamp.max() - g.timestamp.min()).total_seconds()
    expected = span / 10 + 1
    return pd.Series({"expected_rows": int(expected), "actual_rows": len(g),
                       "missing": int(expected) - len(g)})

gap_report = train_raw.groupby(["deviceId", "sessionId"]).apply(expected_vs_actual)
print(f"total dropped packets across the whole training set: {gap_report['missing'].sum()}")
gap_report.head(6)


In [ ]:

# clean: drop glitch rows outright (we can't recover a pinned-rail reading)
clean = train_raw.loc[~glitch_mask].copy()
clean_holdout = holdout_raw.loc[(holdout_raw.moisturePercent >= 0) & (holdout_raw.moisturePercent <= 100)].copy()
print(f"kept {len(clean)}/{len(train_raw)} training rows, {len(clean_holdout)}/{len(holdout_raw)} holdout rows")



### Put the clock back in order

We dropped the glitch rows, which means the time steps are no longer perfectly even —
and every "5 minutes ahead" feature we're about to build assumes an even grid. So we
resample every (device, session) onto a strict 10-second grid first, filling the rare
single missing step by interpolation rather than pretending it was never missing.


In [ ]:

NUMERIC_COLS = ["moistureRaw", "moisturePercent", "pumpOn", "rssi", "uptimeMs"]
STATIC_COLS = ["deviceId", "mode", "dryThreshold", "wetThreshold", "sessionId"]

def to_uniform_grid(g):
    g = g.set_index("timestamp").sort_index()
    numeric = g[NUMERIC_COLS].resample("10s").mean().interpolate(limit=3)
    static = g[STATIC_COLS].resample("10s").ffill()
    return numeric.join(static).dropna().reset_index()

def rebuild_on_uniform_grid(df):
    pieces = [to_uniform_grid(g) for _, g in df.groupby(["deviceId", "sessionId"])]
    return pd.concat(pieces, ignore_index=True)

clean_grid = rebuild_on_uniform_grid(clean)
holdout_grid = rebuild_on_uniform_grid(clean_holdout)

print("training grid:", clean_grid.shape, " | holdout grid:", holdout_grid.shape)
clean_grid.head()



---
## Part 4 — Explore the cleaned data

Same idea as the playground, now on genuine logged sessions. Pick a bench and a
session and look at how closely it matches the intuition you built with the sliders —
and where it doesn't.


In [ ]:

def plot_session(deviceId, sessionId):
    g = clean_grid[(clean_grid.deviceId == deviceId) & (clean_grid.sessionId == sessionId)]
    if g.empty:
        print("no data for that combination"); return
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.fill_between(g.timestamp, 0, 100, where=g.pumpOn.astype(bool), color=ACCENT,
                     alpha=0.12, step="post", label="pump ON")
    ax.plot(g.timestamp, g.moisturePercent, color=FOREST, lw=1.3)
    ax.axhline(g.dryThreshold.iloc[0], color="#C0392B", ls="--", lw=1, label="dryThreshold")
    ax.axhline(g.wetThreshold.iloc[0], color="#2471A3", ls="--", lw=1, label="wetThreshold")
    manual = g[g["mode"] == "manual"]
    if len(manual):
        ax.scatter(manual.timestamp, manual.moisturePercent, s=10, color=AMBER, zorder=5,
                   label="manual mode")
    ax.set_ylim(0, 100); ax.set_ylabel("moisture %")
    ax.set_title(f"{deviceId} — {sessionId}")
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

devices = sorted(clean_grid.deviceId.unique())
sessions = sorted(clean_grid.sessionId.unique())
interact(plot_session,
         deviceId=widgets.Dropdown(options=devices, description="bench"),
         sessionId=widgets.Dropdown(options=sessions, description="session"));


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

clean_grid["moisturePercent"].hist(bins=40, ax=axes[0], color=ACCENT, edgecolor="white")
axes[0].set_title("Distribution of moisturePercent"); axes[0].set_xlabel("moisture %")

for dev, g in clean_grid.groupby("deviceId"):
    axes[1].scatter(g.moistureRaw, g.moisturePercent, s=3, alpha=0.3, label=dev)
axes[1].set_xlabel("moistureRaw"); axes[1].set_ylabel("moisturePercent")
axes[1].set_title("raw \u2192 percent: each bench calibrates slightly differently")
axes[1].legend(markerscale=3, fontsize=8)
plt.tight_layout(); plt.show()



That second panel is the whole reason **Part 5** exists: the same raw ADC value means
a different moisture percentage depending on which bench read it. That's not noise —
it's per-device wiring and probe variance, and it's exactly the kind of thing a model
can absorb that a single hard-coded formula in firmware can't.



---
## Part 5 — Model A: learning a sensor's calibration

**Input:** `moistureRaw`, `deviceId` → **Output:** `moisturePercent`

This is the simplest regression problem in the notebook, and a genuinely useful one:
if you swap a probe or wire a new bench, could a model recalibrate it for you instead
of you hand-tuning constants in `config.h`?


In [ ]:

from sklearn.model_selection import train_test_split

X_raw = clean_grid[["moistureRaw"]].copy()
X_raw_dev = pd.get_dummies(clean_grid[["moistureRaw", "deviceId"]], columns=["deviceId"])
y_cal = clean_grid["moisturePercent"]

Xr_tr, Xr_te, y_tr, y_te = train_test_split(X_raw, y_cal, test_size=0.25, random_state=7)
Xrd_tr, Xrd_te, _, _ = train_test_split(X_raw_dev, y_cal, test_size=0.25, random_state=7)

model_raw_only = LinearRegression().fit(Xr_tr, y_tr)
model_raw_device = LinearRegression().fit(Xrd_tr, y_tr)

pred_a = model_raw_only.predict(Xr_te)
pred_b = model_raw_device.predict(Xrd_te)

print(f"raw only            R\u00b2={r2_score(y_te, pred_a):.4f}   MAE={mean_absolute_error(y_te, pred_a):.2f}%")
print(f"raw + deviceId       R\u00b2={r2_score(y_te, pred_b):.4f}   MAE={mean_absolute_error(y_te, pred_b):.2f}%")


In [ ]:

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_te, pred_a, s=6, alpha=0.3, color=MUTED, label="raw only")
ax.scatter(y_te, pred_b, s=6, alpha=0.3, color=ACCENT, label="raw + deviceId")
lims = [0, 100]
ax.plot(lims, lims, "k--", lw=1, label="perfect prediction")
ax.set_xlabel("actual moisture %"); ax.set_ylabel("predicted moisture %")
ax.set_title("Adding deviceId tightens the fit around the diagonal")
ax.legend(); plt.show()



The gap between those two R² scores *is* the calibration drift you saw in the scatter
plot above, now quantified. Knowing which bench a reading came from measurably
improves a moisture estimate — a small result, but a real one, and it's the same
shape of problem as every "our five machines report slightly different sensor
baselines" situation you'll hit on a real factory floor.



---
## Part 6 — Feature engineering: teaching the model about time

Calibration was a snapshot problem — one reading in, one reading out. Forecasting is
different: to predict where moisture will *be*, the model needs a sense of where it's
been and what's currently pulling it up or down.


In [ ]:

HORIZON_STEPS = 30      # 30 x 10s = 5 minutes ahead
ROLL_WINDOW = "300s"    # 5-minute rolling window

def engineer_features(df):
    out = []
    for (dev, sess), g in df.groupby(["deviceId", "sessionId"]):
        g = g.sort_values("timestamp").set_index("timestamp")
        g["moisture_roll_mean"] = g["moisturePercent"].rolling(ROLL_WINDOW).mean()
        g["moisture_roll_std"] = g["moisturePercent"].rolling(ROLL_WINDOW).std().fillna(0)
        g["moisture_roc_2min"] = (g["moisturePercent"] - g["moisturePercent"].shift(12)) / 2.0
        pump_change = g["pumpOn"].diff().fillna(0) != 0
        change_times = g.index.to_series().where(pump_change).ffill()
        g["minutes_since_pump_toggle"] = (
            (g.index - change_times).dt.total_seconds() / 60.0
        ).fillna(999).clip(upper=60)
        hour = g.index.hour + g.index.minute / 60.0
        g["hour_sin"] = np.sin(2 * np.pi * hour / 24)
        g["hour_cos"] = np.cos(2 * np.pi * hour / 24)
        g["target_future_moisture"] = g["moisturePercent"].shift(-HORIZON_STEPS)
        g["deviceId"], g["sessionId"] = dev, sess
        out.append(g.reset_index())
    result = pd.concat(out, ignore_index=True)
    return result.dropna(subset=["moisture_roll_mean", "moisture_roc_2min", "target_future_moisture"])

FEATURES = ["moisturePercent", "pumpOn", "moisture_roll_mean", "moisture_roll_std",
            "moisture_roc_2min", "minutes_since_pump_toggle", "hour_sin", "hour_cos",
            "dryThreshold", "wetThreshold"]
TARGET = "target_future_moisture"

feat_train = engineer_features(clean_grid)
feat_holdout = engineer_features(holdout_grid)
print("engineered training rows:", len(feat_train))
feat_train[["deviceId", "timestamp"] + FEATURES + [TARGET]].head()



Notice `deviceId` is **not** in `FEATURES`. Unlike Part 5, we want this model to work
on a bench it's never met — bench-06 shows up only in the holdout file, never in
training — so it can only learn from signals that transfer across hardware:
the moisture trend itself, whether the pump is running, and the time of day. Anything
device-specific has to earn its way in through those, not through a lookup table.



### Before we train: what goes in, what comes out

The diagram below is the entire model in one picture — every input on the left,
one number out the right.


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.axis("off")
ax.set_xlim(0, 10); ax.set_ylim(0, 10)

# input boxes
for i, feat in enumerate(FEATURES):
    y = 9.3 - i * 0.95
    ax.add_patch(mpatches.FancyBboxPatch((0.2, y - 0.32), 3.1, 0.62,
                 boxstyle="round,pad=0.05", fc=SAGE, ec=FOREST, lw=1))
    ax.text(1.75, y, feat, ha="center", va="center", fontsize=9.5, color=FOREST, family="monospace")
    ax.annotate("", xy=(4.6, 5.0), xytext=(3.35, y),
                arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8, alpha=0.6))

# model box
ax.add_patch(mpatches.FancyBboxPatch((4.6, 3.9), 2.6, 2.2, boxstyle="round,pad=0.08",
             fc=FOREST, ec=FOREST, lw=1))
ax.text(5.9, 5.35, "MODEL", ha="center", color="white", fontsize=13, fontweight="bold")
ax.text(5.9, 4.75, "Random Forest\nRegressor", ha="center", color=SAGE, fontsize=10)
ax.text(5.9, 4.15, "(baseline: Linear,\npersistence)", ha="center", color=SAGE, fontsize=8)

# output box
ax.annotate("", xy=(9.6, 5.0), xytext=(7.2, 5.0),
            arrowprops=dict(arrowstyle="-|>", color=ACCENT, lw=2.2))
ax.add_patch(mpatches.FancyBboxPatch((7.35, 4.35), 2.45, 1.3, boxstyle="round,pad=0.06",
             fc=ACCENT, ec=FOREST, lw=1))
ax.text(8.58, 5.25, "OUTPUT", ha="center", color=FOREST, fontsize=11, fontweight="bold")
ax.text(8.58, 4.72, "moisture % in\n5 minutes", ha="center", color=FOREST, fontsize=9.5)

ax.set_title("Ten inputs, one forecasting model, one number out", fontsize=12, color=FOREST)
plt.show()



---
## Part 7 — Model B: forecasting moisture 5 minutes ahead

### Why a *time-based* split, not a random one

A random split would put a row from 10:04:10 in training and the row ten seconds
later, from 10:04:20, in the test set. Those two rows share almost the same rolling
mean and the same trend — the model would be "predicting" something it has
practically already seen. That inflates every metric and hides how the model would
really perform on a session it hasn't lived through yet.

So instead: **sessions from 18 Aug and 25 Aug train the model; the 1 Sept session, for
every bench, is the test set** — a session the model has genuinely never seen.


In [ ]:

train_sessions = ["2026-08-18-s1", "2026-08-25-s1"]
test_sessions = ["2026-09-01-s1"]

Xb_train = feat_train[feat_train.sessionId.isin(train_sessions)][FEATURES]
yb_train = feat_train[feat_train.sessionId.isin(train_sessions)][TARGET]
Xb_test = feat_train[feat_train.sessionId.isin(test_sessions)][FEATURES]
yb_test = feat_train[feat_train.sessionId.isin(test_sessions)][TARGET]

print("train rows:", len(Xb_train), " | test rows:", len(Xb_test))


In [ ]:

# baseline: naive persistence — "moisture in 5 minutes = moisture right now"
baseline_pred = Xb_test["moisturePercent"]

linreg = LinearRegression().fit(Xb_train, yb_train)
linreg_pred = linreg.predict(Xb_test)

forest = RandomForestRegressor(n_estimators=250, max_depth=10, random_state=7, n_jobs=-1)
forest.fit(Xb_train, yb_train)
forest_pred = forest.predict(Xb_test)

def report(name, y_true, y_pred):
    return dict(model=name,
                MAE=round(mean_absolute_error(y_true, y_pred), 3),
                RMSE=round(mean_squared_error(y_true, y_pred) ** 0.5, 3),
                R2=round(r2_score(y_true, y_pred), 4))

results = pd.DataFrame([
    report("persistence (naive baseline)", yb_test, baseline_pred),
    report("linear regression", yb_test, linreg_pred),
    report("random forest", yb_test, forest_pred),
]).set_index("model")
results



Look closely at that table before moving on — **plain linear regression should come out
*worse* than just guessing "nothing will change."** That's not a bug in this notebook,
and it's worth sitting with, because it's a real and common failure mode:

Moisture doesn't move in one straight line. While the pump is on it rises at close to
a constant rate; the moment it switches off, it starts decaying back down. One global
straight-line coefficient per feature can't represent "the effect of time flips sign
depending on `pumpOn`" — a linear model has to compromise between those two regimes
and ends up wrong in both. A random forest doesn't have that constraint: it can split
on `pumpOn` first and learn a completely different rule for each branch. That's the
actual reason we reached for a forest and not just a better-tuned straight line —
**the shape of the physics decided the shape of the model**, not habit.


In [ ]:

fig, ax = plt.subplots(figsize=(6.5, 6.5))
sample = np.random.choice(len(yb_test), size=min(1200, len(yb_test)), replace=False)
ax.scatter(yb_test.values[sample], baseline_pred.values[sample], s=8, alpha=0.25, color=MUTED, label="persistence")
ax.scatter(yb_test.values[sample], forest_pred[sample], s=8, alpha=0.35, color=ACCENT, label="random forest")
lims = [yb_test.min() - 2, yb_test.max() + 2]
ax.plot(lims, lims, "k--", lw=1)
ax.set_xlabel("actual moisture % (5 min ahead)"); ax.set_ylabel("predicted")
ax.set_title("Random forest tracks the diagonal much more tightly than \u201cassume nothing changes\u201d")
ax.legend(); plt.show()


In [ ]:

importances = pd.Series(forest.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
importances.plot.barh(ax=ax, color=ACCENT)
ax.set_title("What the forest actually pays attention to")
ax.set_xlabel("importance")
plt.tight_layout(); plt.show()



Expect `pumpOn`, `moisture_roc_2min`, and `moisture_roll_mean` near the top — which
matches physical intuition exactly: the single strongest signal for "where will
moisture be in 5 minutes" is "where is it now and is the pump running." The model
didn't need to be told the physics; it recovered them from the data.

**Try it yourself:** change `HORIZON_STEPS` in the cell above from `30` to `6` (1
minute ahead) or `90` (15 minutes ahead), re-run Part 6 and Part 7, and watch the
random forest's R² change. Shorter horizons are easier to predict — there's less time
for the pump to flip states in between.



---
## Part 8 — Does it work on a bench it has never seen?

Every session so far, train or test, came from bench-01 through bench-05. bench-06 has
been sitting untouched in `telemetry-holdout.csv` this whole time — different wiring,
a different pump flow rate, a different evaporation rate. This is the real test of
whether the model learned **irrigation physics** or just **five specific benches**.


In [ ]:

Xb_holdout = feat_holdout[FEATURES]
yb_holdout = feat_holdout[TARGET]
holdout_baseline = feat_holdout["moisturePercent"]
holdout_pred = forest.predict(Xb_holdout)

holdout_results = pd.DataFrame([
    report("persistence (naive baseline)", yb_holdout, holdout_baseline),
    report("random forest (unseen bench-06)", yb_holdout, holdout_pred),
]).set_index("model")
holdout_results


In [ ]:

g = feat_holdout[feat_holdout.sessionId == feat_holdout.sessionId.iloc[0]].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(g.timestamp, g.target_future_moisture, color=FOREST, lw=1.4, label="actual (5 min ahead)")
ax.plot(g.timestamp, forest.predict(g[FEATURES]), color=ACCENT, lw=1.4, ls="--", label="predicted")
ax.set_ylabel("moisture %"); ax.set_title("bench-06 — a kit the model has never trained on")
ax.legend(); plt.show()



If the random forest's R² on bench-06 lands close to its score on the 1 Sept test
sessions, that's the payoff: the model generalised to new hardware because it was
built on transferable signals (trend, pump state, time of day) rather than on
`deviceId` lookup. That's the same reason we deliberately excluded `deviceId` back in
Part 6 — a forecasting model meant to run on the *next* bench a cohort builds has to
work this way.



---
## Part 9 — Play with the trained model directly

Every slider below is a live input to the random forest you just trained. Move one,
watch the predicted moisture bar move, and try to predict *which direction* it'll move
before you look — that's the real test of whether Part 7's feature-importance chart
actually sunk in.


In [ ]:

def live_predict(moisture_now, pump_on, roll_mean, roll_std, roc_2min,
                  minutes_since_toggle, hour, dry_th, wet_th):
    hour_sin, hour_cos = np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24)
    row = pd.DataFrame([{
        "moisturePercent": moisture_now, "pumpOn": pump_on,
        "moisture_roll_mean": roll_mean, "moisture_roll_std": roll_std,
        "moisture_roc_2min": roc_2min, "minutes_since_pump_toggle": minutes_since_toggle,
        "hour_sin": hour_sin, "hour_cos": hour_cos,
        "dryThreshold": dry_th, "wetThreshold": wet_th,
    }])[FEATURES]
    pred = forest.predict(row)[0]

    fig, ax = plt.subplots(figsize=(8, 1.6))
    ax.barh(0, 100, color="#EDEDED", height=0.5)
    ax.barh(0, moisture_now, color=MUTED, height=0.5, alpha=0.55, label="now")
    ax.barh(0, pred, color=ACCENT, height=0.22, label="predicted, +5 min")
    ax.axvline(dry_th, color="#C0392B", ls="--", lw=1)
    ax.axvline(wet_th, color="#2471A3", ls="--", lw=1)
    ax.set_xlim(0, 100); ax.set_yticks([])
    ax.set_title(f"predicted moisture in 5 minutes: {pred:.1f}%   (now: {moisture_now:.0f}%,"
                 f" change: {pred - moisture_now:+.1f} pts)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=2, fontsize=8, frameon=False)
    plt.show()

interact(
    live_predict,
    moisture_now=widgets.FloatSlider(value=45, min=8, max=92, step=1, description="moisture now"),
    pump_on=widgets.ToggleButtons(options=[("OFF", 0), ("ON", 1)], description="pump"),
    roll_mean=widgets.FloatSlider(value=45, min=8, max=92, step=1, description="5-min mean"),
    roll_std=widgets.FloatSlider(value=1.0, min=0, max=8, step=0.2, description="5-min std"),
    roc_2min=widgets.FloatSlider(value=0.0, min=-3, max=3, step=0.1, description="2-min trend"),
    minutes_since_toggle=widgets.FloatSlider(value=5, min=0, max=60, step=1, description="min since toggle"),
    hour=widgets.FloatSlider(value=13, min=0, max=23.9, step=0.5, description="hour of day"),
    dry_th=widgets.IntSlider(value=35, min=10, max=50, description="dryThreshold"),
    wet_th=widgets.IntSlider(value=65, min=50, max=90, description="wetThreshold"),
);



**A few things worth actually trying:**
- Set `pump ON` with a strongly negative `2-min trend` — you told the model moisture
  was falling *and* that the pump just started. Watch which signal wins.
- Push `min since toggle` high with the pump ON — a pump that's been running a long
  time should be closer to saturating than one that just switched on.
- Set `hour of day` to 13 (early afternoon) vs 3 (middle of the night) with everything
  else identical — the model picked up the evaporation cycle from `hour_sin`/`hour_cos`
  without ever being told what "afternoon" means physically.



---
## Wrap-up

| | Model A — calibration | Model B — forecasting |
|---|---|---|
| Input | `moistureRaw`, `deviceId` | trend, pump state, time of day, thresholds |
| Output | `moisturePercent` right now | `moisturePercent` 5 minutes from now |
| Learns | per-device sensor drift | irrigation dynamics |
| Generalises across benches? | no — needs `deviceId` | yes — checked on bench-06 |

Two honest regression problems, same dataset, opposite design decision about whether
`deviceId` belongs in the feature list — and that decision was driven entirely by what
each model needs to generalise to. That's the actual skill this lab is teaching: not
"how to call `.fit()`", but how to decide what a model is and isn't allowed to lean on.

**Where this connects back to the bench:** your `dryThreshold`/`wetThreshold` logic
*is* a two-input, hand-tuned version of Model B's job — "will moisture cross a line
soon?" A forecasting model doesn't replace that rule, but it's the natural next step
if you ever wanted the thresholds themselves to adapt to the weather instead of being
fixed constants in `config.h`.
